Implemented Ridge Regression from scratch using Batch Gradient Descent with L2 regularization on the diabetes dataset. The custom MeraRidgeGDRegressor class includes feature scaling, proper handling of intercept (not regularized), and gradient descent optimization. After tuning, learning_rate=1.5 achieved R²=0.4625, exactly matching scikit-learn's Ridge performance. A critical fix was scaling test data with the same scaler (.transform()), which improved R² from 0.041 to 0.462


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes

In [404]:
X,y = load_diabetes(return_X_y=True)

In [406]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge,SGDRegressor

In [408]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=4)

In [410]:
X_train.shape
X_test.shape

(89, 10)

In [412]:
reg= Ridge(alpha=0.001,max_iter=500,solver='sparse_cg')

In [414]:
reg.fit(X_train,y_train)

Ridge(alpha=0.001, max_iter=500, solver='sparse_cg')

In [416]:
y_pred=reg.predict(X_test)

In [418]:
print("R2 score",r2_score(y_test,y_pred))
print("intercept b:",reg.intercept_)
print("Coeff ",reg.coef_)

R2 score 0.46250101621910966
intercept b: 151.10198522135005
Coeff  [  34.52193893 -290.84082896  482.4018325   368.0678841  -852.44871106
  501.59162397  180.11113982  270.76336652  759.73536846   37.49137396]


In [420]:
sgd_reg =SGDRegressor(eta0=0.1,learning_rate='constant',max_iter=500,alpha=0.001)

In [434]:
sgd_reg.fit(X_train,y_train)

SGDRegressor(alpha=0.001, eta0=0.1, learning_rate='constant', max_iter=500)

In [436]:
y_pred1 = sgd_reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred1))
print("intercept b:",sgd_reg.intercept_)
print("Coeff ",sgd_reg.coef_)

R2 score 0.42914158094363375
intercept b: [142.98913706]
Coeff  [  49.53405438 -155.68706593  372.20658928  270.57525079   -5.88330437
  -58.66437111 -167.77070461  137.61055074  330.6783784   100.81737225]


In [446]:
from sklearn.preprocessing import StandardScaler
class MeraRidgeGDRegressor:

    def __init__(self,epochs,learning_rate,alpha):
        
        self.epochs= epochs
        self.learning_rate= learning_rate
        self.alpha= alpha
        self.intercept_ = None
        self.coef_ = None
        self.scaler= StandardScaler()
        
    def fit(self,X_train,y_train):
        X_train = self.scaler.fit_transform(X_train)
        self.coef_ = np.ones(X_train.shape[1])
        self.intercept_ = 0
        
        X_train = np.insert(X_train,0,1,axis=1)
        m = X_train.shape[0]
        theta = np.insert(self.coef_,0,self.intercept_)
      
        for i in range(self.epochs):
            
             # Predictions
            y_pred = np.dot(X_train,theta)
            # Error
            error = y_pred - y_train
             # Gradient of MSE
            theta_der = (1/m) * np.dot(X_train.T, error)
            
            # Ridge regularization term
            # Do not regularize intercept
            regularization =self.alpha * np.copy(theta)
            regularization[0]= 0
            # Add Ridge penalty
            theta_der = theta_der +  regularization
            # Update weights
            theta= theta - self.learning_rate * theta_der
                          
        self.intercept_ = theta[0]
        self.coef_ = theta[1:]
        return self 
      
    def predict(self,X_test):
        X_test = self.scaler.transform(X_test)
        return np.dot(X_test,self.coef_) + self.intercept_


In [448]:
reg1 = MeraRidgeGDRegressor(learning_rate=0.1,alpha=0.001,epochs=500)

In [450]:
reg1.fit(X_train,y_train)

In [452]:
y_pred2 = reg1.predict(X_test)
print("R2 score",r2_score(y_test,y_pred2))
print("intercept b:",reg1.intercept_)
print("Coeff ",reg1.coef_)

R2 score 0.466518246891341
intercept b: 151.63172804532567
Coeff  [  1.8519747  -13.63236947  23.60026731  17.57037619 -17.1981273
   5.78031845  -2.094964     9.26149574  28.3190447    1.62164747]
